# Frequency-controlled mechanistic readout — Run 2: ALGO + BW only

Llama-3.1-8B and Qwen2.5-7B: **4-bit NF4**. Qwen2.5-1.5B: **unquantized** (bf16 if supported, else fp16 on T4).

**This run does not queue GSM.** Existing GSM results live in `colab_out/mech_freq_controlled.csv` and must not be overwritten. This notebook writes:

`colab_out/mech_freq_controlled_algo_bw.csv`

**Items** (canonical + W3 only):

- **ALGO:** frozen 61-problem adversarial pool (34 SP + 10 CC + 17 WIS). Not the full 110-ID bank.
- **BW:** 65 PlanBench bank IDs.

Gold is answer **content**, not format scaffolding (Appendix H): ALGO cost/count/total; BW first action word. Families where more than half the golds are the same token are flagged **degenerate** and Wilcoxon is not reported.

`RESUME=True` skips completed `(model, family, problem_id, variant)` rows in the ALGO+BW CSV.

**Secrets:** `HF_TOKEN` (gated Llama).


In [ ]:
# Colab T4: bitsandbytes for quantized loads. Restart the runtime if
# bitsandbytes was just installed and the kernel has not picked it up.
import sys
import subprocess
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.44",
        "accelerate>=0.33",
        "bitsandbytes>=0.43",
        "pandas",
        "scipy",
        "networkx",
        "tqdm",
        "huggingface_hub",
    ]
)


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# ── knobs ────────────────────────────────────────────────────────────────
# Set LIMIT to an int for a smoke test (e.g. 2 items per family). None = full run.
LIMIT = None
DRY_RUN = False          # True: skip GPU, write placeholder rows (pipeline check)
RESUME = True

# Private GitHub clone (Colab secret GITHUB_TOKEN, or env). Public clone works
# without a token. If this notebook is already inside the repo, clone is skipped.
REPO_URL = os.environ.get(
    "RVC_REPO_URL",
    "https://github.com/Adya6714/retrieval-vs-computation.git",
)
REPO_COMMIT = os.environ.get("RVC_REPO_COMMIT", "")  # empty = default branch HEAD

def _secret(name: str) -> str:
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name) or ""
    except Exception:
        return ""

HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN")
GH_TOKEN = _secret("GITHUB_TOKEN")

# Llama-3.1-8B-Instruct is gated: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as _hf_exc:
        print("[setup] huggingface login skipped:", _hf_exc)

def _looks_like_repo(p: Path) -> bool:
    return (p / "probes" / "contamination" / "verify.py").is_file() and (
        p / "data" / "problems" / "question_bank_gsm.csv"
    ).is_file()

def _find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if _looks_like_repo(cand):
            return cand
    colab = Path("/content/retrieval-vs-computation")
    if _looks_like_repo(colab):
        return colab
    return colab

REPO_ROOT = _find_repo()
if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    url = REPO_URL
    if GH_TOKEN and "github.com" in url and url.startswith("https://"):
        url = url.replace("https://", f"https://{GH_TOKEN}@")
    print(f"[setup] cloning {REPO_URL} → {REPO_ROOT}")
    cmd = ["git", "clone", "--depth", "1", url, str(REPO_ROOT)]
    subprocess.check_call(cmd)
    if REPO_COMMIT:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REPO_COMMIT])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPO_COMMIT])

assert _looks_like_repo(REPO_ROOT), (
    f"Could not find probes/ + question banks under {REPO_ROOT}. "
    "Clone the retrieval-vs-computation repo, or set RVC_REPO_URL / GITHUB_TOKEN."
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT_DIR = Path("/content/colab_out") if Path("/content").exists() else (REPO_ROOT / "colab_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[setup] REPO_ROOT={REPO_ROOT}")
print(f"[setup] OUT_DIR={OUT_DIR}")
print(f"[setup] LIMIT={LIMIT} DRY_RUN={DRY_RUN} RESUME={RESUME}")


## IDs, Appendix-N prompt, content-gold (not format keywords)


In [ ]:
from __future__ import annotations

import csv
import gc
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import stats
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

PROBE1_TEMPLATE = (
    "Solve the following problem exactly and provide only the final answer "
    "in the required output format. Problem: {problem}. Format instruction: "
    "{family_specific_output_format}."
)
GSM_FORMAT = (
    "Write the final numerical answer on its own line as #### <number>. "
    "No other text after that tag."
)
ALGO_FORMAT = (
    "Follow the problem's required output format exactly "
    "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
)
BW_FORMAT = (
    "A numbered list of actions only. Each action must be one of the "
    "permitted operators with their arguments. No explanation."
)
FAMILY_FORMAT = {"GSM": GSM_FORMAT, "ALGO": ALGO_FORMAT, "BW": BW_FORMAT}

# Frozen ALGO adversarial pool (rebuild/FROZEN_FILTERS.md). Not bank instance_type.
ALGO_ADV = {
    "CC": [f"CC_{i:02d}" for i in range(1, 11)],
    "SP": [
        "SP_003", "SP_004", "SP_005", "SP_019", "SP_020", "SP_021", "SP_023",
        "SP_024", "SP_026", "SP_027", "SP_028", "SP_029", "SP_030", "SP_037",
        "SP_038", "SP_039", "SP_040", "SP_042", "SP_044", "SP_045", "SP_046",
        "SP_047", "SP_048", "SP_062", "SP_063", "SP_064", "SP_065", "SP_066",
        "SP_068", "SP_069", "SP_070", "SP_071", "SP_072", "SP_073",
    ],
    "WIS": [
        "WIS_003", "WIS_004", "WIS_013", "WIS_014", "WIS_015", "WIS_016",
        "WIS_017", "WIS_018", "WIS_019", "WIS_020", "WIS_023", "WIS_024",
        "WIS_025", "WIS_026", "WIS_027", "WIS_028", "WIS_029",
    ],
}
ALGO_ADV_IDS = ALGO_ADV["CC"] + ALGO_ADV["SP"] + ALGO_ADV["WIS"]
assert len(ALGO_ADV_IDS) == 61, len(ALGO_ADV_IDS)

# Tokens that are format scaffolding, not answer content (Appendix H confound).
FORMAT_KEYWORDS = {
    "path", "count", "selected", "coins", "scoops", "total", "answer",
    "final", "####", "#", ":", "[", "]", "{", "}", ",",
    "path:", "count:", "selected:", "coins:", "scoops:",
}

CLAUDE_P1 = REPO_ROOT / "results/raw/GSM_P1_behavioral_claude.csv"  # GSM already computed; not queued
GSM_BANK = REPO_ROOT / "data/problems/question_bank_gsm.csv"
ALGO_BANK = REPO_ROOT / "data/problems/question_bank_algo.csv"
BW_BANK = REPO_ROOT / "data/problems/question_bank_bw.csv"

GSM_CSV_DO_NOT_TOUCH = (
    Path("/content/colab_out/mech_freq_controlled.csv")
    if Path("/content").exists()
    else (OUT_DIR / "mech_freq_controlled.csv")
)
print(f"[gsm] leave untouched: {GSM_CSV_DO_NOT_TOUCH}  exists={GSM_CSV_DO_NOT_TOUCH.exists()}")


def _norm_bank(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str).fillna("")
    df["problem_id"] = df["problem_id"].astype(str).str.strip()
    vt_col = "variant_type" if "variant_type" in df.columns else "variant"
    df["variant_type"] = df[vt_col].astype(str).str.strip()
    df.loc[df.variant_type.str.lower() == "canonical", "variant_type"] = "canonical"
    return df


gsm_df = _norm_bank(GSM_BANK)
algo_df = _norm_bank(ALGO_BANK)
bw_df = _norm_bank(BW_BANK)
# aliases used by the unigram counter
bank, algo_bank, bw_bank = gsm_df, algo_df, bw_df


def _paired_ids(df: pd.DataFrame) -> list[str]:
    can = set(df.loc[df.variant_type == "canonical", "problem_id"])
    w3 = set(df.loc[df.variant_type == "W3", "problem_id"])
    return sorted(can & w3)


# Frozen adversarial 61 (not all 110 unique ALGO bank IDs). GSM is not queued.
_bank_algo_unique = sorted(algo_df["problem_id"].unique())
ALGO_IDS = [pid for pid in ALGO_ADV_IDS if pid in set(_paired_ids(algo_df))]
BW_IDS = _paired_ids(bw_df)
assert len(ALGO_IDS) == 61, len(ALGO_IDS)
assert len(BW_IDS) == 65, f"expected 65 BW bank IDs, got {len(BW_IDS)}"
print(f"[banks] ALGO={len(ALGO_IDS)} (frozen adv; bank unique={len(_bank_algo_unique)}) BW={len(BW_IDS)}")
print("[banks] GSM_IDS not used for this run")


def bank_row_from(df: pd.DataFrame, pid: str, vt: str, label: str) -> pd.Series:
    sub = df[(df.problem_id == pid) & (df.variant_type == vt)]
    if sub.empty:
        raise KeyError(f"{pid}/{vt} missing from {label} bank")
    return sub.iloc[0]


def gsm_gold_content(correct_answer: str) -> str:
    """Numeric answer-content span. Never #### / Path / Count scaffolding."""
    s = str(correct_answer).strip()
    s = re.sub(r"^####\s*", "", s)
    s = s.replace(",", "")
    try:
        f = float(s)
        if f == int(f):
            return str(int(f))
        return str(f)
    except ValueError:
        m = re.findall(r"-?\d+(?:\.\d+)?", s)
        if not m:
            raise ValueError(f"no numeric gold in {correct_answer!r}")
        return m[-1]


def algo_gold_content(problem_id: str, correct_answer: str) -> str:
    """Answer-content numeric token: SP cost, CC count, WIS total.

    Not Path / Coins / Selected (Appendix H format-keyword confound).
    W3 coin-change uses ``Total:`` instead of ``Count:``.
    """
    s = str(correct_answer)
    pid = str(problem_id).strip().upper()
    if pid.startswith("SP"):
        m = re.search(r"Cost\s*:\s*(-?\d+)", s, flags=re.I)
        if not m:
            raise ValueError(f"{problem_id}: no Cost: gold in {s!r}")
        return m.group(1)
    if pid.startswith("CC"):
        m = re.search(r"(?:Count|Total)\s*:\s*(-?\d+)", s, flags=re.I)
        if not m:
            raise ValueError(f"{problem_id}: no Count:/Total: gold in {s!r}")
        return m.group(1)
    if pid.startswith("WIS"):
        m = re.search(r"Total\s*:\s*(-?\d+)", s, flags=re.I)
        if not m:
            raise ValueError(f"{problem_id}: no Total: gold in {s!r}")
        return m.group(1)
    raise ValueError(f"{problem_id}: unknown ALGO subtype")


def bw_gold_content(correct_answer: str) -> str:
    """First action word of the gold plan (pick-up / unstack / attack / …)."""
    lines = [ln.strip() for ln in str(correct_answer).splitlines() if ln.strip()]
    if not lines:
        raise ValueError("empty BW gold plan")
    line = re.sub(r"^\d+[\.)]\s*", "", lines[0])
    m = re.match(r"([A-Za-z][A-Za-z0-9_-]*)", line)
    if not m:
        raise ValueError(f"no action word in {lines[0]!r}")
    return m.group(1)


def build_user(problem_text: str, family: str = "GSM") -> str:
    return PROBE1_TEMPLATE.format(
        problem=str(problem_text).strip(),
        family_specific_output_format=FAMILY_FORMAT[family],
    )


# Run 2 queue: ALGO + BW only. GSM is not queued and will not be recalculated.
ITEMS: list[dict] = []

for pid in ALGO_IDS:
    for vt in ("canonical", "W3"):
        r = bank_row_from(algo_df, pid, vt, "ALGO")
        ITEMS.append(
            {
                "family": "ALGO",
                "problem_id": pid,
                "variant": vt,
                "problem_text": str(r["problem_text"]),
                "correct_answer": str(r["correct_answer"]),
                "gold_content": algo_gold_content(pid, r["correct_answer"]),
            }
        )

for pid in BW_IDS:
    for vt in ("canonical", "W3"):
        r = bank_row_from(bw_df, pid, vt, "BW")
        ITEMS.append(
            {
                "family": "BW",
                "problem_id": pid,
                "variant": vt,
                "problem_text": str(r["problem_text"]),
                "correct_answer": str(r["correct_answer"]),
                "gold_content": bw_gold_content(r["correct_answer"]),
            }
        )

if LIMIT is not None:
    keep = {("ALGO", pid) for pid in ALGO_IDS[:LIMIT]} | {("BW", pid) for pid in BW_IDS[:LIMIT]}
    ITEMS = [x for x in ITEMS if (x["family"], x["problem_id"]) in keep]

print(
    f"[queue] {len(ITEMS)} items "
    f"({len({x['problem_id'] for x in ITEMS if x['family']=='ALGO'})} ALGO + "
    f"{len({x['problem_id'] for x in ITEMS if x['family']=='BW'})} BW × 2 variants)"
)
assert not any(x["family"] == "GSM" for x in ITEMS), "GSM must not be in the Run 2 queue"
for fam in ("ALGO", "BW"):
    sub = [x for x in ITEMS if x["family"] == fam]
    vc = Counter(x["gold_content"] for x in sub)
    can_vc = Counter(x["gold_content"] for x in sub if x["variant"] == "canonical")
    if vc:
        modal, n_m = vc.most_common(1)[0]
        frac = n_m / len(sub)
        can_modal, can_n = can_vc.most_common(1)[0]
        can_n_ids = sum(can_vc.values())
        can_frac = can_n / max(can_n_ids, 1)
        flag = " DEGENERATE" if (frac > 0.5 or can_frac > 0.5) else ""
        print(
            f"  {fam}: modal={modal!r} {n_m}/{len(sub)}={frac:.1%}; "
            f"canonical {can_modal!r} {can_n}/{can_n_ids}={can_frac:.1%}{flag}"
        )


## Token targeting + unigram frequency proxy

Target id is the **first BPE of the gold content that would follow the chat prompt** (joint encode), not an isolated `'5'` vs `' 5'` mismatch.

Frequency proxy (no Pile dump on Colab):

1. Tokenize every GSM + ALGO + BW bank `problem_text` + `correct_answer` with **this model's tokenizer** → unigram counts.
2. Also record `token_id` (BPE id rank) as a secondary proxy.

Terciles are computed **within (model, family)** over unique canonical gold-token unigram counts.

A family is **degenerate** if more than half of its golds (all items, or canonical-only) are the same token. Wilcoxon is not reported for a degenerate family.


In [ ]:
def wrap_chat(tokenizer, user_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        add_generation_prompt=True,
        tokenize=False,
    )


def resolve_target_token(tokenizer, prompt: str, answer: str) -> tuple[int, str, list[int], str]:
    """Prompt-aware first token of `answer` after `prompt` (same as mechanistic scripts)."""
    def enc(text: str) -> list[int]:
        return tokenizer.encode(text, add_special_tokens=False)

    prompt_ids = enc(prompt)
    answer_ids_bare = enc(answer)
    candidates: list[tuple[str, int, int, list[int]]] = []
    for sep in ("", " "):
        joint = enc(prompt + sep + answer)
        if len(joint) <= len(prompt_ids):
            continue
        if joint[: len(prompt_ids)] != prompt_ids:
            continue
        rest = joint[len(prompt_ids) :]
        candidates.append((sep, int(rest[0]), len(joint), rest))
    if not candidates:
        if not answer_ids_bare:
            return -1, "", [], "EMPTY"
        tid = int(answer_ids_bare[0])
        return tid, tokenizer.decode([tid]), answer_ids_bare, "FALLBACK"
    candidates.sort(key=lambda c: c[2])
    sep, tid, _, rest = candidates[0]
    return tid, tokenizer.decode([tid]), rest, repr(sep)


def assert_content_gold(decoded: str, family: str) -> None:
    d = decoded.strip().lower()
    compact = d.replace(" ", "")
    if compact in FORMAT_KEYWORDS or d in FORMAT_KEYWORDS:
        raise AssertionError(f"gold token is a format keyword: {decoded!r}")
    if family in {"GSM", "ALGO"}:
        if not re.search(r"\d", decoded):
            raise AssertionError(
                f"{family} content-gold token must contain a digit, got {decoded!r}"
            )


def bank_unigram_counter(tokenizer) -> Counter:
    """Training-proxy: question-bank token unigrams (this tokenizer)."""
    c: Counter = Counter()
    for df in (bank, algo_bank, bw_bank):
        for _, r in df.iterrows():
            text = f"{r['problem_text']}\n{r['correct_answer']}"
            ids = tokenizer.encode(str(text), add_special_tokens=False)
            c.update(ids)
    return c


UNQUANTIZED = {
    "Qwen/Qwen2.5-1.5B-Instruct",  # fits unquantized on T4; 7B/8B stay nf4
}


def load_model(model_id: str):
    """4-bit NF4 for 7B/8B; unquantized bf16 (fp16 fallback) for the 1.5B."""
    assert torch.cuda.is_available(), "GPU required (Colab T4)."
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    if model_id in UNQUANTIZED:
        if torch.cuda.is_bf16_supported():
            dtype = torch.bfloat16
            dtype_label = "bfloat16"
        else:
            dtype = torch.float16
            dtype_label = "float16"
            print(
                f"[model] {model_id}: GPU lacks native bf16 "
                f"(T4 is sm_75); loading unquantized {dtype_label}"
            )
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=dtype,
            token=HF_TOKEN or True,
        )
        quant_label = f"unquantized {dtype_label}"
    else:
        dtype = torch.float16
        dtype_label = "float16"
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
        )
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb,
            device_map="auto",
            torch_dtype=dtype,
            token=HF_TOKEN or True,
        )
        quant_label = "nf4 4-bit bitsandbytes (double quant)"

    mdl.eval()
    device = next(mdl.parameters()).device
    n_layers = int(mdl.config.num_hidden_layers)
    print(f"[model] {model_id}  {quant_label}  layers={n_layers} device={device}")
    return tok, mdl, device, n_layers, dtype, quant_label


def unload(mdl):
    del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.inference_mode()
def readout_layers(model, tokenizer, device, user_text: str, gold_content: str, family: str) -> dict:
    prompt = wrap_chat(tokenizer, user_text)
    tid, decoded, gold_ids, sep_note = resolve_target_token(tokenizer, prompt, gold_content)
    assert_content_gold(decoded, family)
    if DRY_RUN or model is None:
        n = 32
        if model is not None:
            n = int(getattr(model.config, "num_hidden_layers", 32))
        return {
            "gold_token_id": tid,
            "gold_token_decoded": decoded,
            "gold_token_ids": [int(x) for x in gold_ids],
            "sep_note": "DRY_RUN",
            "n_layers": n,
            "ranks": [1] * n,
            "logprobs": [0.0] * n,
            "cosines": [0.0] * n,
        }
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    out = model(**inputs, output_hidden_states=True, use_cache=False)
    hidden_states = out.hidden_states[1:]  # skip embedding; layer 0 = first block
    W_U = model.lm_head.weight.detach().float()  # [vocab, d]
    u = W_U[tid]
    u_norm = F.normalize(u.unsqueeze(0), dim=-1)
    ranks, logprobs, cosines = [], [], []
    for layer_h in hidden_states:
        h = layer_h[0, -1, :].float()
        logits = h @ W_U.T
        target_logit = logits[tid]
        rank = int((logits > target_logit).sum().item()) + 1
        lp = float(F.log_softmax(logits, dim=-1)[tid].item())
        cos = float(F.cosine_similarity(h.unsqueeze(0), u_norm, dim=-1).item())
        ranks.append(rank)
        logprobs.append(round(lp, 6))
        cosines.append(round(cos, 6))
    n_layers = len(ranks)
    del out, hidden_states, inputs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        "gold_token_id": int(tid),
        "gold_token_decoded": decoded,
        "gold_token_ids": [int(x) for x in gold_ids],
        "sep_note": sep_note,
        "n_layers": n_layers,
        "ranks": ranks,
        "logprobs": logprobs,
        "cosines": cosines,
    }


## Hugging Face login (gated Llama)

Run this **before** loading Llama. Uses `HF_TOKEN` from Colab secrets if set; otherwise prompts.


In [ ]:
from huggingface_hub import login, get_token

_tok = HF_TOKEN or get_token()
if _tok:
    login(token=_tok, add_to_git_credential=False)
    print("[hf] authenticated")
else:
    login()


## Run models (one at a time — T4 16GB)


In [ ]:
MODELS = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
]

MECH_CSV = OUT_DIR / "mech_freq_controlled_algo_bw.csv"
MECH_COLS = [
    "family", "model", "problem_id", "variant", "layer",
    "rank", "logprob", "cosine_to_gold_unembed",
    "gold_content", "gold_token_id", "gold_token_decoded",
    "gold_unigram_count", "gold_token_id_rank_proxy",
    "n_layers", "sep_note",
]

# Never write to the GSM file.
assert MECH_CSV.name != "mech_freq_controlled.csv"
print(f"[out] ALGO+BW -> {MECH_CSV}")
print(f"[out] GSM file left as {GSM_CSV_DO_NOT_TOUCH}")

done: set[tuple[str, str, str, str]] = set()
write_header = True
if RESUME and MECH_CSV.exists() and MECH_CSV.stat().st_size > 0:
    prev = pd.read_csv(MECH_CSV, dtype=str)
    if "family" not in prev.columns:
        prev.insert(0, "family", prev["problem_id"].map(
            lambda p: "ALGO" if str(p).split("_")[0] in {"CC", "SP", "WIS"} else "BW"
        ))
        prev.to_csv(MECH_CSV, index=False)
    done = set(
        zip(
            prev["model"].astype(str),
            prev["family"].astype(str),
            prev["problem_id"].astype(str),
            prev["variant"].astype(str),
        )
    )
    write_header = False
    print(f"[resume] {len(done)} completed (model, family, problem_id, variant) keys")
elif MECH_CSV.exists() and not RESUME:
    MECH_CSV.unlink()

FREQ_BY_MODEL: dict[str, Counter] = {}
N_LAYERS_BY_MODEL: dict[str, int] = {}
QUANT_BY_MODEL: dict[str, str] = {}
DTYPE_BY_MODEL: dict[str, str] = {}
DTYPE_USED = "float16"
QUANT = "mixed: Llama-3.1-8B + Qwen2.5-7B = nf4 4-bit; Qwen2.5-1.5B = unquantized (see quantization_by_model)"

for model_id in MODELS:
    pending = [
        it for it in ITEMS
        if (model_id, it["family"], it["problem_id"], it["variant"]) not in done
    ]
    print(f"\n===== {model_id}  pending={len(pending)}/{len(ITEMS)} =====")
    if DRY_RUN:
        class _Dummy:
            config = type("c", (), {"num_hidden_layers": 32})
        tok = AutoTokenizer.from_pretrained("gpt2")  # tiny, dry-run only
        mdl, device, n_layers = None, "cpu", 32
        # still want a real tokenizer for gold-token / unigram if possible
        try:
            tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True)
        except Exception as exc:
            print("[dry-run] tokenizer fallback gpt2:", exc)
        FREQ_BY_MODEL[model_id] = bank_unigram_counter(tok)
        N_LAYERS_BY_MODEL[model_id] = n_layers
        QUANT_BY_MODEL[model_id] = "DRY_RUN"
        DTYPE_BY_MODEL[model_id] = "n/a"
    else:
        tok, mdl, device, n_layers, dtype, quant_label = load_model(model_id)
        FREQ_BY_MODEL[model_id] = bank_unigram_counter(tok)
        N_LAYERS_BY_MODEL[model_id] = n_layers
        QUANT_BY_MODEL[model_id] = quant_label
        DTYPE_BY_MODEL[model_id] = str(dtype).replace("torch.", "")

    freq = FREQ_BY_MODEL[model_id]
    with MECH_CSV.open("a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=MECH_COLS)
        if write_header:
            w.writeheader()
            write_header = False
        for it in tqdm(pending, desc=model_id.split("/")[-1]):
            user = build_user(it["problem_text"], it["family"])
            metrics = readout_layers(mdl, tok, device, user, it["gold_content"], it["family"])
            tid = metrics["gold_token_id"]
            uni = int(freq.get(tid, 0))
            for layer_i in range(metrics["n_layers"]):
                w.writerow(
                    {
                        "family": it["family"],
                        "model": model_id,
                        "problem_id": it["problem_id"],
                        "variant": it["variant"],
                        "layer": layer_i,
                        "rank": metrics["ranks"][layer_i],
                        "logprob": metrics["logprobs"][layer_i],
                        "cosine_to_gold_unembed": metrics["cosines"][layer_i],
                        "gold_content": it["gold_content"],
                        "gold_token_id": tid,
                        "gold_token_decoded": metrics["gold_token_decoded"],
                        "gold_unigram_count": uni,
                        "gold_token_id_rank_proxy": tid,  # lower id ≈ more frequent in many BPEs
                        "n_layers": metrics["n_layers"],
                        "sep_note": metrics["sep_note"],
                    }
                )
            f.flush()
            done.add((model_id, it["family"], it["problem_id"], it["variant"]))

    if not DRY_RUN:
        unload(mdl)
    del tok
    gc.collect()

print(f"wrote {MECH_CSV}")

# Drive copy as soon as the sweep finishes (survives a later crash).
_drive_dir = Path("/content/drive/MyDrive/rvc_colab_out")
if Path("/content").exists():
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
        except Exception as exc:
            print("[drive] mount skipped:", exc)
    try:
        _drive_dir.mkdir(parents=True, exist_ok=True)
        if MECH_CSV.exists():
            import shutil as _shutil
            _dst = _drive_dir / MECH_CSV.name
            _shutil.copy2(MECH_CSV, _dst)
            print(f"[backup] {MECH_CSV} -> {_dst}  ({_dst.stat().st_size} bytes)")
    except Exception as exc:
        print("[backup] skipped:", exc)


## Summary: canonical vs W3 final-layer rank, Wilcoxon, frequency terciles


In [ ]:
df = pd.read_csv(MECH_CSV)
if "family" not in df.columns:
    df["family"] = df["problem_id"].map(
        lambda p: "GSM" if str(p).startswith("GSM") else (
            "ALGO" if str(p).split("_")[0] in {"CC", "SP", "WIS"} else "BW"
        )
    )
df["rank"] = pd.to_numeric(df["rank"], errors="coerce")
df["gold_unigram_count"] = pd.to_numeric(df["gold_unigram_count"], errors="coerce")
df["layer"] = pd.to_numeric(df["layer"], errors="coerce")

# final layer per (model, family, problem, variant)
final = (
    df.sort_values("layer")
    .groupby(["model", "family", "problem_id", "variant"], as_index=False)
    .tail(1)
)

print("=== median final-layer rank ===")
print(
    final.groupby(["model", "family", "variant"])["rank"]
    .median()
    .unstack("variant")
    .to_string()
)

DEGEN_FRAC = 0.5


def degeneracy(toks: pd.Series) -> tuple[bool, str, int, int]:
    s = toks.astype(str).str.strip()
    n = int(len(s))
    if n == 0:
        return True, "", 0, 0
    vc = s.value_counts()
    modal = str(vc.index[0])
    n_m = int(vc.iloc[0])
    return (n_m / n) > DEGEN_FRAC, modal, n_m, n


print("\n=== gold-token degeneracy (Appendix H) ===")
degen_keys: set[tuple[str, str]] = set()
audit_rows = []
for (model_id, fam), g in final.groupby(["model", "family"]):
    uniq = g.drop_duplicates(["problem_id", "variant"])
    can = uniq[uniq.variant == "canonical"]
    dec_col = "gold_token_decoded"
    all_d, all_m, all_n, all_N = degeneracy(uniq[dec_col])
    can_d, can_m, can_n, can_N = degeneracy(can[dec_col])
    flag = all_d or can_d
    if flag:
        degen_keys.add((model_id, fam))
    why = []
    if all_d:
        why.append(f"all-items {all_m!r}={all_n}/{all_N}")
    if can_d:
        why.append(f"canonical {can_m!r}={can_n}/{can_N}")
    audit_rows.append(
        {
            "model": model_id,
            "family": fam,
            "degenerate": flag,
            "modal_all": all_m,
            "share_all": f"{all_n}/{all_N}",
            "modal_canonical": can_m,
            "share_canonical": f"{can_n}/{can_N}",
            "note": ("DEGENERATE: " + "; ".join(why) + " — Wilcoxon not reported")
            if flag
            else "ok",
        }
    )
audit = pd.DataFrame(audit_rows)
print(audit.to_string(index=False))

# terciles of gold unigram frequency (canonical gold, within model × family)
can_freq = final[final.variant == "canonical"][
    ["model", "family", "problem_id", "gold_unigram_count"]
].drop_duplicates()

def tercile_labels(s: pd.Series) -> pd.Series:
    # qcut can collapse if many ties (e.g. lots of '0'); fall back to rank-based
    try:
        return pd.qcut(s, 3, labels=["low", "mid", "high"], duplicates="drop")
    except ValueError:
        return pd.qcut(s.rank(method="first"), 3, labels=["low", "mid", "high"])

parts = []
for (model_id, fam), g in can_freq.groupby(["model", "family"]):
    t = g.copy()
    t["freq_tercile"] = tercile_labels(t["gold_unigram_count"])
    parts.append(t)
terc = pd.concat(parts, ignore_index=True) if parts else can_freq.assign(freq_tercile="mid")
paired = final.merge(
    terc[["model", "family", "problem_id", "freq_tercile"]],
    on=["model", "family", "problem_id"],
    how="left",
)

wide = paired.pivot_table(
    index=["model", "family", "problem_id", "freq_tercile"],
    columns="variant",
    values="rank",
    aggfunc="first",
).reset_index()
wide = wide.dropna(subset=["canonical", "W3"])


def wilcoxon_block(
    sub: pd.DataFrame,
    model_id: str,
    fam: str,
    tercile: str,
    *,
    degenerate: bool = False,
    degen_note: str = "",
) -> dict:
    label = f"{model_id} | {fam} | {tercile}"
    a = sub["canonical"].to_numpy(dtype=float)
    b = sub["W3"].to_numpy(dtype=float)
    base = {
        "slice": label,
        "model": model_id,
        "family": fam,
        "freq_tercile": tercile,
        "n_pairs": int(len(sub)),
    }
    if degenerate:
        return {
            **base,
            "median_rank_canonical": float("nan"),
            "median_rank_W3": float("nan"),
            "median_can_minus_W3": float("nan"),
            "W": float("nan"),
            "p_two_sided": float("nan"),
            "note": degen_note or "DEGENERATE — result not reported",
        }
    if len(a) < 3 or np.allclose(a, b):
        stat, p = float("nan"), float("nan")
        note = "n<3 or identical"
    else:
        try:
            stat, p = stats.wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
            note = "ok"
        except ValueError as exc:
            stat, p, note = float("nan"), float("nan"), str(exc)
    return {
        **base,
        "median_rank_canonical": float(np.median(a)),
        "median_rank_W3": float(np.median(b)),
        "median_can_minus_W3": float(np.median(a - b)),
        "W": stat,
        "p_two_sided": p,
        "note": note,
    }


rows = []
for (model_id, fam), g in wide.groupby(["model", "family"]):
    flag = (model_id, fam) in degen_keys
    degen_note = ""
    if flag:
        hit = audit[(audit.model == model_id) & (audit.family == fam)]
        degen_note = str(hit.iloc[0]["note"]) if len(hit) else "DEGENERATE — result not reported"
        print(f"\n[skip] {model_id} / {fam}: {degen_note}")
    rows.append(
        wilcoxon_block(g, model_id, fam, "all", degenerate=flag, degen_note=degen_note)
    )
    for tname, sg in g.groupby("freq_tercile", observed=False):
        rows.append(
            wilcoxon_block(
                sg, model_id, fam, f"tercile={tname}",
                degenerate=flag, degen_note=degen_note,
            )
        )

wtab = pd.DataFrame(rows)
print("\n=== paired Wilcoxon (final-layer rank, canonical vs W3) ===")
print(wtab.to_string(index=False))
wtab.to_csv(OUT_DIR / "mech_freq_controlled_algo_bw_summary.csv", index=False)

print("\n=== gold token audit (must not be format keywords) ===")
dec = df.drop_duplicates(["model", "family", "problem_id", "variant"])[
    ["model", "family", "gold_token_decoded", "gold_content", "gold_unigram_count"]
]
for fam, sg in dec.groupby("family"):
    print(f"\n-- {fam} --")
    print(sg["gold_token_decoded"].value_counts().head(10).to_string())
bad = dec[dec["gold_token_decoded"].str.strip().str.lower().isin(FORMAT_KEYWORDS)]
print(f"\nformat-keyword golds: {len(bad)} (expect 0)")
assert len(bad) == 0, bad.head()


## Manifest


In [ ]:
import subprocess

def git_hash() -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
        ).strip()
    except Exception as exc:
        return f"unavailable ({exc})"

n_items = int(df.drop_duplicates(["model", "family", "problem_id", "variant"]).shape[0] / max(len(MODELS), 1)) if "df" in dir() else len(ITEMS)

manifest = {
    "notebook": "mechanistic_frequency_controlled.ipynb",
    "model_string": MODELS,
    "dtype": DTYPE_BY_MODEL if "DTYPE_BY_MODEL" in dir() else DTYPE_USED,
    "quantization": QUANT_BY_MODEL if "QUANT_BY_MODEL" in dir() else (QUANT if not DRY_RUN else "DRY_RUN"),
    "quantization_confound": (
        "Qwen2.5-1.5B-Instruct is unquantized (bf16 if supported, else fp16 on T4). "
        "Llama-3.1-8B-Instruct and Qwen2.5-7B-Instruct are nf4 4-bit. "
        "Do not treat rank differences as a pure size effect."
    ),
    "decoding_config": {
        "readout": "forward pass only (no generation); last prompt position",
        "chat_template": True,
        "prompt": "Appendix N Probe-1 family-specific format instruction",
        "gold": {
            "ALGO": "SP Cost / CC Count|Total / WIS Total (not Path/Coins/Selected)",
            "BW": "first action word of gold plan",
        },
    },
    "n_items": n_items,
    "n_ids": {
        "ALGO": len({x["problem_id"] for x in ITEMS if x["family"] == "ALGO"}),
        "BW": len({x["problem_id"] for x in ITEMS if x["family"] == "BW"}),
    },
    "n_models": len(MODELS),
    "id_source": {
        "ALGO": "frozen adversarial pool 34 SP + 10 CC + 17 WIS = 61",
        "BW": "data/problems/question_bank_bw.csv canonical IDs (n=65)",
        "GSM": "NOT QUEUED — existing mech_freq_controlled.csv left untouched",
    },
    "frequency_proxy": "GSM+ALGO+BW bank unigram counts under each model's tokenizer; token_id as secondary rank proxy",
    "degeneracy_rule": "family flagged if modal gold token is >50% of all items or of canonical items; Wilcoxon not reported",
    "git_commit_hash": git_hash(),
    "output_csv": str(MECH_CSV),
    "n_layers_by_model": {k: int(v) for k, v in N_LAYERS_BY_MODEL.items()} if "N_LAYERS_BY_MODEL" in dir() else {},
}
print("=== MANIFEST ===")
print(json.dumps(manifest, indent=2))
(OUT_DIR / "mech_freq_controlled_algo_bw_manifest.json").write_text(json.dumps(manifest, indent=2))

# Drive + laptop download for the three ALGO/BW artifacts. Does not touch GSM CSV.
_out_files = [
    OUT_DIR / "mech_freq_controlled_algo_bw.csv",
    OUT_DIR / "mech_freq_controlled_algo_bw_summary.csv",
    OUT_DIR / "mech_freq_controlled_algo_bw_manifest.json",
]
_drive_dir = Path("/content/drive/MyDrive/rvc_colab_out")
if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] mount skipped:", exc)
try:
    import shutil as _shutil
    _drive_dir.mkdir(parents=True, exist_ok=True)
    for p in _out_files:
        if p.exists():
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
except Exception as exc:
    print("[backup] skipped:", exc)
try:
    from google.colab import files as _colab_files  # type: ignore
    for p in _out_files:
        if p.exists():
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
except Exception as exc:
    print("[download] skipped (not Colab or download blocked):", exc)
